# 01 - Model check

Before running anything I need to know the model is what I think it is. This notebook
checks four things and runs no experiments:

1. it loads and matches its published size
2. it grows
3. it contains the ethylene glycol pathway
4. glycolate really does have only one way out

The model is iJN1463, a genome-scale reconstruction of *Pseudomonas putida* KT2440
(Nogales et al. 2020), from https://bigg.bio/models/iJN1463.

**This notebook finds a chokepoint, so it is worth saying up front why one would matter.** Bao et al. are upcycling: they
want the carbon from PET ending up in a product such as mcl-PHA. For that goal, carbon
spilled as glycolate is carbon lost from the product.

But glycolate accumulating in the medium is a problem even if you only want the monomers
gone. It is a weak acid, so a standing pool acidifies the culture, and it is carbon the
cell has taken in and failed to process, so the culture stalls while it clears. Bao's own
data shows the glycolate pool is transient, meaning it does get cleared eventually, but
clearing it costs time.

So resolving the chokepoint helps whether the aim is a product or plain monomer removal.
That is worth stating because it means the findings here do not depend on which of those
two goals a reader cares about.

In [1]:
import sys
sys.path.append("../src")
from putida import *

import pandas as pd

model = load_model()

print("model id      ", model.id)
print("reactions     ", len(model.reactions))
print("metabolites   ", len(model.metabolites))
print("genes         ", len(model.genes))
print("objective     ", [r.id for r in model.reactions if r.objective_coefficient != 0])

model id       iJN1463
reactions      2929
metabolites    2155
genes          1462
objective      ['BIOMASS_KT2440_WT3']


## Does it grow?

BiGG models come with a default medium. Solving straight away is the cheapest check that
nothing is broken: a reconstruction with a gap in a biomass pathway returns zero.

One trap here. Listing the medium by its bounds is misleading, because iJN1463 leaves a
pile of odd exchanges open at high limits that the cell never touches. What I want is
what it actually *consumes*, which is the exchange reactions carrying negative flux. That
also lets me add the carbon up and check it against the carbon source.

In [2]:
sol = model.optimize()
print("status      ", sol.status)
print("growth rate ", round(sol.objective_value, 5), "per hour\n")

print("what it actually consumes:")
total_C = 0.0
for r in sorted(model.exchanges, key=lambda r: sol.fluxes[r.id]):
    f = sol.fluxes[r.id]
    if f >= -1e-6:
        continue
    met = model.metabolites.get_by_id(r.id.replace("EX_", "", 1))
    nC = met.elements.get("C", 0)
    total_C += -f * nC
    print(f"   {r.id:<26} {f:>9.3f}   C={nC:<3} {met.name[:30]}")

glc = -sol.fluxes[EX_GLUCOSE]
print(f"\ntotal carbon in: {total_C:.2f} mmol C/gDW/h")
print(f"glucose alone:   {glc * 6:.2f} mmol C/gDW/h")
print("\nThose match, so growth is limited by glucose and not by something the")
print("default medium left open. (A trace of the PQQ precursor peptide also gets")
print("taken up, but it rounds away.)")

MW = 180.16 / 1000.0
print(f"\nyield: {sol.objective_value / glc:.4f} gDW/mmol = "
      f"{sol.objective_value / glc / MW:.3f} gDW per g glucose")
print("That's in the usual range for P. putida. It is a sanity check, not a")
print("validation, until I compare it to the number in Nogales et al.")

status       optimal
growth rate  0.58612 per hour

what it actually consumes:
   EX_o2_e                      -11.423   C=0   dioxygen
   EX_nh4_e                      -6.239   C=0   ammonium
   EX_glc__D_e                   -6.000   C=6   D-glucopyranose
   EX_pi_e                       -0.539   C=0   hydrogenphosphate
   EX_so4_e                      -0.109   C=0   sulfate
   EX_k_e                        -0.093   C=0   potassium(1+)
   EX_fe2_e                      -0.009   C=0   iron(2+)
   EX_mg2_e                      -0.004   C=0   magnesium(2+)
   EX_ca2_e                      -0.002   C=0   calcium(2+)
   EX_cl_e                       -0.002   C=0   chloride
   EX_na1_e                      -0.002   C=0   sodium(1+)
   EX_mobd_e                     -0.002   C=0   molybdate
   EX_cobalt2_e                  -0.002   C=0   cobalt(2+)
   EX_cu2_e                      -0.002   C=0   copper(2+)
   EX_ni2_e                      -0.002   C=0   nickel(2+)
   EX_zn2_e                  

## The ethylene glycol pathway

Four steps. Printing them confirms the route exists and fixes the reaction IDs, which
are not the obvious ones: iJN1463 prefixes anything added to the generic BiGG set, so
ethylene glycol is `__iJN1463__glycol_e` and searching for "ethylene" finds nothing.

In [3]:
EG_PATHWAY = [
    (R_EG_OXIDASE, "ethylene glycol -> glycolaldehyde",        "pedE/pedH, periplasmic, uses PQQ"),
    (R_GCALDD,     "glycolaldehyde -> glycolate",              "makes NADH"),
    (R_GLCDEF,     "glycolate -> glyoxylate",                  "glcD/E/F, the operon Bao overexpressed"),
    (R_GCL,        "2 glyoxylate -> tartronate semialdehyde",  "gcl, derepressed in Pp-E by deleting gclR"),
]

for rid, step, note in EG_PATHWAY:
    r = model.reactions.get_by_id(rid)
    print(f"{rid}")
    print(f"   {step}")
    print(f"   {r.reaction}")
    print(f"   genes: {r.gene_reaction_rule}")
    print(f"   {note}\n")

__iJN1463__GLYCOLDHpp
   ethylene glycol -> glycolaldehyde
   __iJN1463__glycol_p + pqq_p --> gcald_p + pqqh2_p
   genes: PP_2674 or PP_2679
   pedE/pedH, periplasmic, uses PQQ

GCALDD
   glycolaldehyde -> glycolate
   gcald_c + h2o_c + nad_c --> glyclt_c + 2.0 h_c + nadh_c
   genes: PP_2680 or PP_0545
   makes NADH

GLYCTO1
   glycolate -> glyoxylate
   glyclt_c + o2_c --> glx_c + h2o2_c
   genes: PP_3745 and PP_3746 and PP_3747
   glcD/E/F, the operon Bao overexpressed

GLXCL
   2 glyoxylate -> tartronate semialdehyde
   2.0 glx_c + h_c --> 2h3opp_c + co2_c
   genes: PP_4297
   gcl, derepressed in Pp-E by deleting gclR



## The chokepoint

This is the cell the rest of the project depends on.

A metabolite with several consumers has routes around a blockage. One with a single
consumer does not: cap that reaction and the metabolite has nowhere to go but out of the
cell.

Glycolate having one consumer means nothing on its own, though, because it could just
mean iJN1463 is drawn sparsely around there. So I count consumers for three neighbours
as well.

In [4]:
for met_id, label in [("glyclt_c", "glycolate"),
                      ("glx_c",    "glyoxylate (control)"),
                      ("pyr_c",    "pyruvate (control)"),
                      ("succ_c",   "succinate (control)")]:
    consumers = consumers_of(model, met_id)
    print(f"{label:<24} {len(consumers):>3} consuming reaction(s)")
    if len(consumers) <= 3:
        for rid, rxn, gpr in consumers:
            print(f"      {rid:<14} {rxn}")
            print(f"      {'':14} genes: {gpr}")
    print()

rule, _ = genes_for(model, R_GLCDEF)
print(f"Glycolate disposal runs through {R_GLCDEF} alone, encoded by {rule}.")
print("Those are glcD, glcE and glcF: the operon Bao put under a strong promoter")
print("to stop glycolate building up.")

glycolate                  1 consuming reaction(s)
      GLYCTO1        glyclt_c + o2_c --> glx_c + h2o2_c
                     genes: PP_3745 and PP_3746 and PP_3747

glyoxylate (control)       5 consuming reaction(s)

pyruvate (control)        13 consuming reaction(s)

succinate (control)        3 consuming reaction(s)
      SUCDi          q8_c + succ_c --> fum_c + q8h2_c
                     genes: PP_4190 and PP_4191 and PP_4192 and PP_4193
      PPCSCT         ppcoa_c + succ_c --> prpnte_c + succoa_c
                     genes: PP_0154
      SUCOAS         atp_c + coa_c + succ_c <=> adp_c + pi_c + succoa_c
                     genes: PP_4186 and PP_4185

Glycolate disposal runs through GLYCTO1 alone, encoded by PP_3745 and PP_3746 and PP_3747.
Those are glcD, glcE and glcF: the operon Bao put under a strong promoter
to stop glycolate building up.


## The respiratory chain

Everything later in the project is about oxygen, so these need to be present and to look
like the literature says they should. Two details come back later:

- succinate dehydrogenase is the only TCA step that reduces the quinone pool instead of
  NAD+, which couples succinate handling to respiration
- the glycolate reaction **as written here** consumes O2 directly and makes hydrogen
  peroxide, so clearing glycolate costs oxygen on top of ordinary respiration

Notebook 06 shows that second reaction has the wrong chemistry on the right genes:
*glcD/E/F* encode a dehydrogenase, whose acceptor is not O2, rather than an oxidase.
The oxygen cost survives that correction, because a dehydrogenase reduces the quinone
pool and that pool still has to be drained by oxygen. The route is indirect and the
penalty is smaller, and notebook 07 quantifies both.

In [5]:
for rid in [R_SUCDH, R_NADH16, R_NADH5, R_GLCDEF, R_CATALASE]:
    r = model.reactions.get_by_id(rid)
    print(f"{rid:<12} {r.name}")
    print(f"{'':12} {r.reaction}\n")

print("where electrons finally meet oxygen:")
q8h2 = model.metabolites.get_by_id("q8h2_c")
for r in q8h2.reactions:
    if r.metabolites[q8h2] < 0 and "o2_c" in [m.id for m in r.metabolites]:
        print(f"   {r.id:<14} {r.reaction}")

SUCDi        succinate dehydrogenase (irreversible)
             q8_c + succ_c --> fum_c + q8h2_c

NADH16pp     NADH dehydrogenase (ubiquinone-8 & 3 protons) (periplasm)
             4.0 h_c + nadh_c + q8_c --> 3.0 h_p + nad_c + q8h2_c

NADH5        NADH dehydrogenase (ubiquinone-8 )
             h_c + nadh_c + q8_c --> nad_c + q8h2_c

GLYCTO1      Glycolate oxidase
             glyclt_c + o2_c --> glx_c + h2o2_c

CAT          catalase
             2.0 h2o2_c --> 2.0 h2o_c + o2_c

where electrons finally meet oxygen:
   CYTBDpp        2.0 h_c + 0.5 o2_c + q8h2_c --> h2o_c + 2.0 h_p + q8_c
   CYTBO3_4pp     4.0 h_c + 0.5 o2_c + q8h2_c --> h2o_c + 4.0 h_p + q8_c


## Conclusion

The model loads, is the right size, grows on glucose at a sensible yield, and contains
the whole ethylene glycol route with the right genes on it.

The useful result came from a check rather than an experiment: **glycolate has exactly
one consuming reaction**, `GLYCTO1`, encoded by `PP_3745`, `PP_3746` and `PP_3747`
(*glcD*, *glcE*, *glcF*). Glyoxylate, pyruvate and succinate all have several. So there
is no way around it, and anything that limits that reaction forces glycolate out of the
cell.

Bao et al. picked the same operon as their target, from experiments, without using this
model.

**Limitations.** iJN1463 is wild-type KT2440. Bao's strains are EM42 and M31 derivatives
with a terephthalate plasmid added and *ped* deleted, none of which is here. And the
absence of another glycolate route is evidence about the reconstruction rather than the
organism: a route nobody has annotated wouldn't show up.

**Next.** Notebook 02 measures what the ethylene glycol branch costs to run.